# Deepfake Speech Detection Demo Notebook

This notebook reproduces the main experiment families described in `main.pdf` using the repository's existing scripts and cached artifacts. It is intentionally thin: experiment logic stays in `experiments/scripts`, audits stay in `experiments/axis_audits`, and this notebook orchestrates, validates, and summarizes results for reviewers.

The models used for checkpoint demos are expected under `models/important`. Dataset-loading experiments use Hugging Face through the existing scripts.

## 0. Environment and Notebook-Safe Runner

This cell checks the runtime, sets deterministic/quiet defaults, and defines a small wrapper for running repository scripts from a notebook. The wrapper monkey-patches `sys.argv` because many scripts are command-line programs and otherwise see Jupyter's kernel arguments. The optional `torch.load` patch sets `weights_only=False` for Lightning checkpoints saved with Python objects; PyTorch 2.6+ changed the default and can reject these trusted local research checkpoints.

In [ ]:
from pathlib import Path
import contextlib
import importlib
import json
import os
import runpy
import sys
import time

ROOT = Path.cwd().resolve()
assert (ROOT / 'experiments').exists(), f'Run this notebook from the repo root, got {ROOT}'
EXPERIMENTS = ROOT / 'experiments'
SCRIPTS = EXPERIMENTS / 'scripts'
RESULTS = EXPERIMENTS / 'results'
AUDITS = EXPERIMENTS / 'axis_audits'
MODEL_DIR = ROOT / 'models' / 'important'

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('HF_HOME', str(ROOT / 'data' / 'huggingface'))
os.environ.setdefault('HF_DATASETS_CACHE', str(ROOT / 'data' / 'huggingface' / 'datasets'))
os.environ.setdefault('WANDB_MODE', 'disabled')

def dependency_report():
    mods = ['numpy', 'pandas', 'scipy', 'sklearn', 'torch', 'torchaudio', 'transformers', 'datasets', 'pytorch_lightning']
    rows = []
    for name in mods:
        try:
            mod = importlib.import_module(name)
            rows.append((name, getattr(mod, '__version__', 'installed')))
        except Exception as exc:
            rows.append((name, f'MISSING: {type(exc).__name__}: {exc}'))
    return rows

print('Repo root:', ROOT)
print('Python:', sys.version)
for pkg, ver in dependency_report():
    print(f'{pkg:18s} {ver}')
print('Checkpoints:', [p.name for p in sorted(MODEL_DIR.glob('*.ckpt'))])

def patch_torch_load_for_lightning():
    """Notebook compatibility patch for trusted local Lightning checkpoints."""
    try:
        import torch
    except Exception:
        return
    if getattr(torch.load, '_new_demo_patched', False):
        return
    original = torch.load
    def patched_load(*args, **kwargs):
        kwargs.setdefault('weights_only', False)
        return original(*args, **kwargs)
    patched_load._new_demo_patched = True
    torch.load = patched_load

@contextlib.contextmanager
def notebook_argv(script_path, extra_args=None):
    old_argv = sys.argv[:]
    sys.argv = [str(script_path)] + list(extra_args or [])
    try:
        yield
    finally:
        sys.argv = old_argv

def run_script(rel_path, extra_args=None, required_outputs=()):
    """Run an existing repo script exactly once from this notebook."""
    patch_torch_load_for_lightning()
    script_path = ROOT / rel_path
    assert script_path.exists(), f'Missing script: {script_path}'
    print(f'\n>>> running {rel_path}')
    t0 = time.time()
    with notebook_argv(script_path, extra_args):
        runpy.run_path(str(script_path), run_name='__main__')
    print(f'<<< finished {rel_path} in {time.time() - t0:.1f}s')
    for out in required_outputs:
        out_path = ROOT / out
        assert out_path.exists(), f'Expected output missing after {rel_path}: {out_path}'
    return True

def read_json(rel_path):
    return json.loads((ROOT / rel_path).read_text())

def show_csv(rel_path, n=20):
    import pandas as pd
    df = pd.read_csv(ROOT / rel_path)
    display(df.head(n))
    return df


## 1. Paper Metric Targets

This cell loads the paper-facing metric targets from the checked-in summary artifacts. Later cells compare freshly produced outputs against these values and flag mismatches rather than modifying results.

In [ ]:
EXPECTED = read_json('final_outputs2/summary_assets/key_numbers.json')
print(json.dumps(EXPECTED, indent=2))

def close_enough(actual, expected, tol=5e-4):
    return abs(float(actual) - float(expected)) <= tol

def metric_check(name, actual, expected, tol=5e-4):
    ok = close_enough(actual, expected, tol)
    status = 'OK' if ok else 'MISMATCH'
    print(f'{status:9s} {name:40s} actual={actual:.6f} expected={expected:.6f} tol={tol}')
    return {'metric': name, 'actual': float(actual), 'expected': float(expected), 'tol': tol, 'ok': ok}

metric_results = []


## Experiment: Axis Production and Geometry Law (I2/I3/J1)

Runs the existing geometry-production and LDA audit scripts. This tests the paper claim that per-system hardness is predicted by position/spread along a natural-to-synthetic axis in frozen WavLM-L12 representation space, and that the finding is not merely an arbitrary linear-classifier artifact.

In [ ]:
# I2 recomputes the feature battery; J1 recomputes centroid/LDA/logreg/detector-axis controls.
# I3 artifacts are consumed by several downstream scripts and are already part of the repo's cached experiment layer.
for rel, outs in [
    ('experiments/scripts/i2_geometry_battery.py', ['experiments/results/i2_geometry_battery/utt_features.csv']),
    ('experiments/scripts/j1_lda_and_audit.py', ['experiments/results/j1_lda_audit/j1_stats.json']),
]:
    run_script(rel, required_outputs=outs)

j1 = read_json('experiments/results/j1_lda_audit/j1_stats.json')
print(json.dumps(j1, indent=2)[:4000])


## Experiment: Axis Fusion (I7)

Runs the frozen-axis score fusion experiment. This tests the actionable paper claim that detector logits underuse axis evidence and that system-disjoint fusion reduces EER, especially under domain shift.

In [ ]:
run_script('experiments/scripts/i7_axis_fusion.py', required_outputs=[
    'experiments/results/i7_axis_fusion/i7_headline.csv',
    'experiments/results/i7_axis_fusion/i7_stats.json',
])
i7_headline = show_csv('experiments/results/i7_axis_fusion/i7_headline.csv')
i7_stats = read_json('experiments/results/i7_axis_fusion/i7_stats.json')
metric_results.append(metric_check('MLAAD WavLM-GAT fusion dEER', i7_stats['dEER'], EXPECTED['mlaad_wavlm_gat']['dEER'], tol=0.003))


## Experiment: Adaptive Axis Head (J3)

Runs the deployable calibration-head experiment. This tests the claim that a small labeled calibration set can estimate a corpus-internal axis and improve frozen detector scores without retraining the backbone.

In [ ]:
# This script loads ASVspoof data from Hugging Face via datasets.load_dataset.
run_script('experiments/scripts/j3_axis_adaptive_head.py', required_outputs=[
    'experiments/results/j3_axis_adaptive/j3_results.csv',
    'experiments/results/j3_axis_adaptive/j3_summary.csv',
])
j3 = show_csv('experiments/results/j3_axis_adaptive/j3_summary.csv', n=50)
import pandas as pd
j3_lda250 = j3[(j3['dataset'] == 'mlaad') & (j3['head'] == 'lda') & (j3['n_cal'] == 250)]
if len(j3_lda250):
    actual = float(j3_lda250.iloc[0]['dEER'])
    metric_results.append(metric_check('J3 MLAAD LDA n=250 dEER', actual, EXPECTED['j3_mlaad_lda250_dEER'], tol=0.01))


## Experiment: mini_goat / GOAT Baselines and Head Ablations (E4)

Runs the existing GOAT baseline, head-discovery, and top-head ablation scripts. This tests the paper's WavLM-GAT/GOAT detector family claims and the mechanism claim that specific heads/directions carry relevant evidence. The notebook uses checkpoints from `models/important` and leaves the experiment implementation in `experiments/scripts`.

In [ ]:
# These scripts can be GPU-heavy. They are included for end-to-end reproduction and will reuse cached outputs when scripts support it.
for rel in [
    'experiments/scripts/e4_goat_baseline.py',
    'experiments/scripts/goat_head_discovery.py',
    'experiments/scripts/e4_goat_topheads_ablation.py',
]:
    run_script(rel)

candidate_outputs = [
    'experiments/results/mlaad/head_discovery/comparison_summary.json',
    'experiments/results/mlaad/head_discovery/target_heads.json',
    'experiments/results/mlaad/e4_ablation/mlaad_goat_e4/in_distribution/pooled_summary.json',
]
for rel in candidate_outputs:
    p = ROOT / rel
    print(rel, 'FOUND' if p.exists() else 'not found')
    if p.exists() and p.suffix == '.json':
        print(json.dumps(read_json(rel), indent=2)[:2000])


## Experiment: AASIST Cross-Family and Fine-Tuned Baseline (J5/J6)

Runs the AASIST experiments. This tests architecture generality: the geometry/hardness law and axis fusion should replicate beyond the WavLM-GAT detector family.

In [ ]:
# J5 evaluates official zero-shot AASIST and fusion; J6 evaluates the MLAAD fine-tuned AASIST baseline.
for rel, outs in [
    ('experiments/scripts/j5_aasist_crossfamily.py', ['experiments/results/j5_aasist/j5_results.json']),
    ('experiments/scripts/j6_train_aasist_mlaad.py', ['experiments/results/j6_aasist_mlaad/j6_results.json']),
]:
    run_script(rel, required_outputs=outs)

j5 = read_json('experiments/results/j5_aasist/j5_results.json')
j6 = read_json('experiments/results/j6_aasist_mlaad/j6_results.json')
print('J5:', json.dumps(j5, indent=2)[:4000])
print('J6:', json.dumps(j6, indent=2)[:4000])
metric_results.extend([
    metric_check('AASIST zero-shot MLAAD baseline EER', j5['eer']['mlaad'], EXPECTED['mlaad_aasist_zeroshot']['baseline_eer'], tol=5e-4),
    metric_check('AASIST zero-shot MLAAD fused EER', j5['H4']['mlaad']['EER_fused'], EXPECTED['mlaad_aasist_zeroshot']['fused_eer'], tol=5e-4),
    metric_check('AASIST fine-tuned test EER', j6['eer_test'], EXPECTED['mlaad_aasist_ft']['test_eer'], tol=5e-4),
    metric_check('AASIST fine-tuned sd_along rho', j6['law']['sd_along']['rho'], EXPECTED['sd_along_aasist_ft_rho'], tol=5e-4),
])


## Experiment: ASVspoof21 Prospective Prediction (J4)

Runs the ASVspoof21 prospective prediction script. This tests the paper claim that the LDA axis projection P3 prospectively predicts unseen ASVspoof21 LA attack hardness for both WavLM-GAT and AASIST.

In [ ]:
# Loads ASVspoof data from Hugging Face through the existing script.
run_script('experiments/scripts/j4_asvspoof21_prospective.py', required_outputs=[
    'experiments/results/j4_asvspoof21/j4_results.json',
    'experiments/results/j4_asvspoof21/j4_preregistered_predictions.json',
])
j4 = read_json('experiments/results/j4_asvspoof21/j4_results.json')
print(json.dumps(j4, indent=2))
metric_results.append(metric_check('ASVspoof21 WavLM P3 rho', j4['P3']['rho'], EXPECTED['p3_rho_wavlm_gat'], tol=5e-4))
metric_results.append(metric_check('ASVspoof21 WavLM P3 p', j4['P3']['p'], EXPECTED['p3_p_wavlm_gat'], tol=5e-4))


## Experiment: Existing Audit Suite

Runs the existing red-team audits under `experiments/axis_audits`. This tests whether the reported claims survive independent recomputation, multiplicity checks, leakage checks, and consistency checks. No new audit code is created in this notebook.

In [ ]:
audit_scripts = [
    'audit2_sdalong_claim.py',
    'audit3_asvspoof_prospective.py',
    'audit4_axis_rotation.py',
    'audit5_fusion_claims.py',
    'audit6_itw_speaker.py',
    'audit7_agreement.py',
    'audit8_i1_causal.py',
    'audit9_hardness_reliability.py',
    'audit1_multiplicity.py',
    'audit10_consistency.py',
]
for script in audit_scripts:
    run_script(f'experiments/axis_audits/{script}')

verdict = ROOT / 'experiments/axis_audits/audits_outputs/COMPREHENSIVE_VERDICT.md'
if verdict.exists():
    print(verdict.read_text()[:5000])
else:
    print('Comprehensive verdict not found; run build_verdict_pdf.py if a PDF/roll-up rebuild is required.')


## Checkpoint Demo: Load One Submitted Checkpoint

Demonstrates that one checkpoint loads in the notebook context and exposes a nonempty state dict. This is the notebook-side checkpoint demonstration; the next cell provides an all-checkpoint validation loop for submission QA without making every reviewer rerun heavyweight model inference inside the main experiment path.

In [ ]:
patch_torch_load_for_lightning()
import torch

demo_ckpt = MODEL_DIR / 'mini_goat.ckpt'
assert demo_ckpt.exists(), f'Missing demo checkpoint: {demo_ckpt}'
ckpt = torch.load(demo_ckpt, map_location='cpu')
state = ckpt.get('state_dict', ckpt)
num_tensors = sum(1 for v in state.values() if hasattr(v, 'shape'))
num_params = sum(int(v.numel()) for v in state.values() if hasattr(v, 'numel'))
sample_keys = list(state.keys())[:12]
print('Loaded:', demo_ckpt)
print('Tensor entries:', num_tensors)
print('Parameter count:', num_params)
print('First keys:', sample_keys)
assert num_tensors > 0 and num_params > 0


## Checkpoint QA: Verify Every Submission Checkpoint Loads

Runs a CPU load check across all checkpoints in `models/important`. This intentionally avoids putting all checkpoint loads on the main narrative path, but it should be run before submission. A failure here should be reported, not hidden.

In [ ]:
patch_torch_load_for_lightning()
import hashlib
import pandas as pd
import torch

rows = []
for path in sorted(MODEL_DIR.glob('*.ckpt')):
    row = {'checkpoint': path.name, 'bytes': path.stat().st_size, 'ok': False, 'error': ''}
    try:
        obj = torch.load(path, map_location='cpu')
        sd = obj.get('state_dict', obj)
        row['state_tensors'] = sum(1 for v in sd.values() if hasattr(v, 'shape'))
        row['param_count'] = sum(int(v.numel()) for v in sd.values() if hasattr(v, 'numel'))
        row['sha256_12'] = hashlib.sha256(path.read_bytes()).hexdigest()[:12]
        row['ok'] = row['state_tensors'] > 0 and row['param_count'] > 0
        del obj, sd
    except Exception as exc:
        row['error'] = f'{type(exc).__name__}: {exc}'
    rows.append(row)

ckpt_df = pd.DataFrame(rows)
display(ckpt_df)
failed = ckpt_df[~ckpt_df['ok']]
if len(failed):
    raise RuntimeError('Some checkpoints failed to load: ' + failed[['checkpoint', 'error']].to_string(index=False))


## Final Metric Comparison

Collects the metric checks from the notebook. Any mismatch is a real validation finding and should be reported rather than tuned away.

In [ ]:
import pandas as pd
summary = pd.DataFrame(metric_results)
display(summary)
if len(summary) and not bool(summary['ok'].all()):
    raise AssertionError('One or more paper metric checks mismatched. Inspect the table above.')
print('All collected metric checks matched their paper targets within tolerance.')
